# 🚀 Feature Engineering Masterclass
### From Zero to Production — Complete Guide + Real-World Project

---

**What this notebook covers:**
- Part 1: Foundations — Data Types, EDA, Missing Values, Outliers
- Part 2: Encoding — Label, One-Hot, Ordinal, Target, Frequency, Binary
- Part 3: Scaling & Transformations — StandardScaler, MinMax, RobustScaler, Log, Box-Cox
- Part 4: Feature Creation — Math features, DateTime, Text, Interactions, Aggregations
- Part 5: Feature Selection — Variance, Correlation, Statistical Tests, Tree Importance, RFE, Lasso
- Part 6: 🔥 FULL END-TO-END PROJECT — Customer Churn Prediction with complete pipeline

> **Author note:** Every code block is runnable and self-contained. Read top to bottom — each concept builds on the last.

---

## ⚙️ Setup — Install & Import Everything

In [ ]:
# Install required libraries (run once)
import subprocess, sys
pkgs = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn',
        'category_encoders', 'scipy', 'xgboost', 'imbalanced-learn']
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('✅ All packages installed!')

In [ ]:
# ── Core ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (12, 5)})

# ── Sklearn — Preprocessing ───────────────────────────────────────────────
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    PowerTransformer, QuantileTransformer,
    LabelEncoder, OrdinalEncoder, OneHotEncoder
)

# ── Sklearn — Feature Selection ───────────────────────────────────────────
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest,
    f_classif, mutual_info_classif,
    RFE, RFECV, SelectFromModel
)

# ── Sklearn — Models ──────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression, Lasso, LassoCV, Ridge
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    RandomForestRegressor
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# ── Sklearn — Pipeline & Evaluation ───────────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    StratifiedKFold, GridSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)
from sklearn.impute import SimpleImputer, KNNImputer

# ── Category Encoders ─────────────────────────────────────────────────────
from category_encoders import TargetEncoder, BinaryEncoder

# ── XGBoost ───────────────────────────────────────────────────────────────
from xgboost import XGBClassifier

# ── Scipy ─────────────────────────────────────────────────────────────────
from scipy import stats
from scipy.stats import skew, kurtosis, shapiro, probplot

print('✅ All imports successful!')
print(f'   Pandas  : {pd.__version__}')
print(f'   NumPy   : {np.__version__}')
print(f'   Sklearn : {__import__("sklearn").__version__}')

---
# 📘 PART 1: Foundations — Data Types & EDA
---

## 1.1 Understanding Data Types

Every feature engineering decision starts with knowing what *kind* of data you have.
The five families:

| Family | Sub-type | Example | Arithmetic? |
|--------|----------|---------|-------------|
| Numeric | Continuous | height, salary | ✅ |
| Numeric | Discrete | num_children, page_views | ✅ |
| Categorical | Nominal | color, country | ❌ |
| Categorical | Ordinal | education level, rating | ❌ (only order) |
| DateTime | — | order_date, login_time | ❌ (extract parts) |

In [ ]:
# ── Build a demo dataset with all data type families ──────────────────────
np.random.seed(42)
n = 200

demo_df = pd.DataFrame({
    # Numeric — continuous
    'height_cm':       np.random.normal(170, 10, n).round(1),
    'weight_kg':       np.random.normal(70, 12, n).round(1),
    'salary':          np.random.lognormal(10.8, 0.5, n).round(0),

    # Numeric — discrete
    'num_children':    np.random.randint(0, 5, n),
    'years_exp':       np.random.randint(0, 30, n),

    # Categorical — nominal
    'department':      np.random.choice(['Engineering','Marketing','Sales','HR','Finance'], n),
    'country':         np.random.choice(['USA','India','UK','Germany','Canada'], n,
                                         p=[0.4, 0.25, 0.15, 0.1, 0.1]),

    # Categorical — ordinal
    'education':       np.random.choice(['High School','Bachelor','Master','PhD'], n,
                                         p=[0.2, 0.45, 0.25, 0.1]),
    'satisfaction':    np.random.choice(['Very Low','Low','Medium','High','Very High'], n),

    # DateTime
    'hire_date':       pd.date_range('2015-01-01', periods=n, freq='7D'),

    # Boolean
    'is_manager':      np.random.choice([True, False], n, p=[0.2, 0.8]),

    # Target
    'churned':         np.random.choice([0, 1], n, p=[0.75, 0.25])
})

# Inject some missing values realistically
missing_idx = np.random.choice(n, 20, replace=False)
demo_df.loc[missing_idx[:10], 'salary'] = np.nan
demo_df.loc[missing_idx[10:], 'education'] = np.nan

print(f'Dataset shape: {demo_df.shape}')
demo_df.head()

In [ ]:
# ── Automatic data type profiling ─────────────────────────────────────────
def profile_dtypes(df):
    """Categorise every column by semantic type."""
    numeric_cols     = df.select_dtypes(include=['int64','float64']).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object','category']).columns.tolist()
    bool_cols        = df.select_dtypes(include=['bool']).columns.tolist()
    datetime_cols    = df.select_dtypes(include=['datetime64']).columns.tolist()

    # Flag numeric columns that may actually be categorical (low cardinality)
    suspicious = [c for c in numeric_cols if df[c].nunique() < 10]

    print('=' * 55)
    print(f'  Dataset: {df.shape[0]} rows × {df.shape[1]} columns')
    print('=' * 55)
    print(f'  Numeric     : {numeric_cols}')
    print(f'  Categorical : {categorical_cols}')
    print(f'  Boolean     : {bool_cols}')
    print(f'  Datetime    : {datetime_cols}')
    print(f'  ⚠ Suspicious (low cardinality numeric): {suspicious}')
    print('=' * 55)
    return {
        'numeric': numeric_cols,
        'categorical': categorical_cols,
        'bool': bool_cols,
        'datetime': datetime_cols,
        'suspicious': suspicious
    }

dtype_map = profile_dtypes(demo_df)

## 1.2 Exploratory Data Analysis (EDA)

EDA answers: *What does my data look like?* before you touch a single feature.
Think of it as a medical check-up for your dataset.

In [ ]:
# ── Univariate: Numeric Features ──────────────────────────────────────────
numeric_features = ['height_cm', 'weight_kg', 'salary', 'years_exp']

fig, axes = plt.subplots(2, len(numeric_features), figsize=(18, 8))
fig.suptitle('Univariate Analysis — Numeric Features', fontsize=15, fontweight='bold')

for i, col in enumerate(numeric_features):
    data = demo_df[col].dropna()

    # Histogram + KDE
    axes[0, i].hist(data, bins=25, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0, i].set_title(f'{col}\nskew={skew(data):.2f}', fontweight='bold')
    axes[0, i].set_xlabel('Value')
    axes[0, i].set_ylabel('Count')

    # Box plot (shows outliers)
    axes[1, i].boxplot(data, vert=True, patch_artist=True,
                       boxprops=dict(facecolor='lightblue'))
    axes[1, i].set_title(f'{col} — Box Plot')
    axes[1, i].set_ylabel('Value')

plt.tight_layout()
plt.show()

# Summary stats
print('\n📊 Summary Statistics:')
demo_df[numeric_features].describe().round(2)

In [ ]:
# ── Univariate: Categorical Features ──────────────────────────────────────
cat_features = ['department', 'country', 'education', 'satisfaction']

fig, axes = plt.subplots(1, len(cat_features), figsize=(20, 5))
fig.suptitle('Univariate Analysis — Categorical Features', fontsize=15, fontweight='bold')

for i, col in enumerate(cat_features):
    counts = demo_df[col].value_counts()
    counts.plot(kind='bar', ax=axes[i], color='coral', edgecolor='white')
    axes[i].set_title(f'{col}\n({demo_df[col].nunique()} unique)', fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)

    # Annotate counts
    for bar in axes[i].patches:
        axes[i].annotate(f'{int(bar.get_height())}',
                         (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                         ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Bivariate: Feature vs Target ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Bivariate Analysis — Features vs Target (churned)', fontsize=14, fontweight='bold')

# Numeric vs Target: salary by churn
for label, grp in demo_df.groupby('churned')['salary']:
    axes[0].hist(grp.dropna(), bins=20, alpha=0.6,
                 label=f'Churned={label}', edgecolor='white')
axes[0].set_title('Salary Distribution by Churn')
axes[0].set_xlabel('Salary')
axes[0].legend()

# Categorical vs Target: churn rate by department
churn_by_dept = demo_df.groupby('department')['churned'].mean().sort_values(ascending=False)
churn_by_dept.plot(kind='bar', ax=axes[1], color='tomato', edgecolor='white')
axes[1].set_title('Churn Rate by Department')
axes[1].set_ylabel('Churn Rate')
axes[1].tick_params(axis='x', rotation=30)
axes[1].axhline(demo_df['churned'].mean(), color='navy', linestyle='--', label='Overall rate')
axes[1].legend()

# Numeric vs Target: experience years
demo_df.boxplot(column='years_exp', by='churned', ax=axes[2],
                patch_artist=True)
axes[2].set_title('Years Experience by Churn')
axes[2].set_xlabel('Churned')
plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation Matrix ─────────────────────────────────────────────────────
numeric_data = demo_df[numeric_features + ['churned']].dropna()
corr = numeric_data.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle mask

plt.figure(figsize=(10, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print top correlations with target
print('\n📊 Top correlations with target (churned):')
print(corr['churned'].sort_values(key=abs, ascending=False).drop('churned'))

## 1.3 Missing Value Analysis & Treatment

In [ ]:
# ── Visualise Missingness ─────────────────────────────────────────────────
def plot_missing(df, title='Missing Value Analysis'):
    missing = df.isnull().sum()
    missing_pct = (df.isnull().mean() * 100).round(2)
    missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
    missing_df = missing_df[missing_df['count'] > 0].sort_values('pct', ascending=False)

    if missing_df.empty:
        print('✅ No missing values found!')
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    # Bar chart
    missing_df['pct'].plot(kind='bar', ax=axes[0], color='tomato', edgecolor='white')
    axes[0].set_title('Missing % per Column')
    axes[0].set_ylabel('Missing %')
    axes[0].axhline(5, color='orange', linestyle='--', label='5% threshold')
    axes[0].axhline(30, color='red', linestyle='--', label='30% threshold')
    axes[0].legend()
    axes[0].tick_params(axis='x', rotation=0)

    # Heatmap of missing pattern
    miss_matrix = df[missing_df.index].isnull().astype(int)
    axes[1].imshow(miss_matrix.T, aspect='auto', cmap='Reds', interpolation='none')
    axes[1].set_yticks(range(len(missing_df.index)))
    axes[1].set_yticklabels(missing_df.index)
    axes[1].set_xlabel('Row index')
    axes[1].set_title('Missing Pattern (red = missing)')

    plt.tight_layout()
    plt.show()

    print('\n📋 Missing Value Summary:')
    print(missing_df.to_string())

plot_missing(demo_df)

In [ ]:
# ── Four Imputation Strategies Compared ───────────────────────────────────
df_work = demo_df.copy()

# Strategy 1: Add missing indicator BEFORE imputing (preserves the signal)
df_work['salary_was_missing'] = df_work['salary'].isnull().astype(int)
df_work['education_was_missing'] = df_work['education'].isnull().astype(int)

# Strategy 2: Mean imputation (numeric)
df_work['salary_mean_imputed'] = df_work['salary'].fillna(df_work['salary'].mean())

# Strategy 3: Median imputation (robust to outliers)
df_work['salary_median_imputed'] = df_work['salary'].fillna(df_work['salary'].median())

# Strategy 4: Mode imputation (categorical)
df_work['education_mode_imputed'] = df_work['education'].fillna(
    df_work['education'].mode()[0]
)

# Strategy 5: KNN Imputation (advanced)
knn_imp = KNNImputer(n_neighbors=5)
salary_knn = knn_imp.fit_transform(df_work[['salary_median_imputed', 'years_exp']])
# (KNNImputer needs numeric-only columns)

# Compare imputation methods on salary
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Imputation Strategy Comparison — Salary', fontsize=13, fontweight='bold')

original = df_work['salary'].dropna()
mean_imp  = df_work['salary_mean_imputed']
med_imp   = df_work['salary_median_imputed']

for ax, data, label, color in zip(
    axes,
    [original, mean_imp, med_imp],
    ['Original (with NaN)', 'Mean Imputed', 'Median Imputed'],
    ['steelblue', 'coral', 'green']
):
    ax.hist(data, bins=25, color=color, edgecolor='white', alpha=0.8)
    ax.set_title(label)
    ax.axvline(data.mean(), color='red', linestyle='--', label=f'Mean={data.mean():.0f}')
    ax.legend()

plt.tight_layout()
plt.show()
print('\n✅ Missing indicators added, imputation complete.')

## 1.4 Outlier Detection & Treatment

In [ ]:
# ── Three Outlier Detection Methods ───────────────────────────────────────
def detect_outliers(series, method='iqr'):
    """Returns boolean mask of outliers."""
    s = series.dropna()
    if method == 'iqr':
        Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
        IQR = Q3 - Q1
        return (series < Q1 - 1.5 * IQR) | (series > Q3 + 1.5 * IQR)
    elif method == 'zscore':
        return (np.abs(stats.zscore(series.dropna())) > 3).reindex(series.index).fillna(False)
    elif method == 'modified_zscore':
        median = s.median()
        mad = np.median(np.abs(s - median))
        modified_z = 0.6745 * (series - median) / mad
        return np.abs(modified_z) > 3.5

col = 'salary'
salary = df_work['salary_median_imputed']

iqr_mask    = detect_outliers(salary, 'iqr')
zscore_mask = detect_outliers(salary, 'zscore')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Outlier Detection Methods — Salary', fontsize=13, fontweight='bold')

# Box plot
axes[0].boxplot(salary, vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[0].set_title(f'Box Plot\nIQR outliers: {iqr_mask.sum()}')

# Scatter with IQR outliers
axes[1].scatter(range(len(salary)), salary, c=iqr_mask.map({True:'red', False:'steelblue'}),
                alpha=0.6, s=20)
axes[1].set_title(f'IQR Method (±1.5×IQR)\nOutliers: {iqr_mask.sum()}')
axes[1].set_xlabel('Row index')
axes[1].set_ylabel('Salary')

# Z-score scatter
axes[2].scatter(range(len(salary)), salary, c=zscore_mask.map({True:'red', False:'green'}),
                alpha=0.6, s=20)
axes[2].set_title(f'Z-Score Method (|z|>3)\nOutliers: {zscore_mask.sum()}')
axes[2].set_xlabel('Row index')

plt.tight_layout()
plt.show()

# Winsorization — cap at 5th & 95th percentile
lower, upper = salary.quantile(0.05), salary.quantile(0.95)
df_work['salary_winsorized'] = salary.clip(lower=lower, upper=upper)
print(f'\n🔧 Winsorization applied: capped to [{lower:.0f}, {upper:.0f}]')
print(f'   Before → max: {salary.max():.0f}, min: {salary.min():.0f}')
print(f'   After  → max: {df_work["salary_winsorized"].max():.0f}, '
      f'min: {df_work["salary_winsorized"].min():.0f}')

---
# 📙 PART 2: Categorical Encoding
---

> ML models need numbers. Encoding converts text categories → numbers without introducing false math.

In [ ]:
# ── One-Hot Encoding ──────────────────────────────────────────────────────
# Best for: nominal (no order), low cardinality, linear models

dept_series = demo_df[['department']].copy()

# Method A: pandas get_dummies (quick)
dept_ohe_pandas = pd.get_dummies(dept_series, columns=['department'], drop_first=True)

# Method B: sklearn (preferred for pipelines)
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
dept_ohe_sk = pd.DataFrame(
    ohe.fit_transform(dept_series),
    columns=ohe.get_feature_names_out()
)

print('Original values:', dept_series['department'].unique())
print('\nAfter One-Hot Encoding (drop_first=True):')
print(dept_ohe_pandas.head(8))
print('\n💡 Why drop_first=True?')
print('   If Engineering=0, Marketing=0, Sales=0, HR=0 → must be Finance')
print('   So Finance column is redundant (dummy variable trap)')

In [ ]:
# ── Ordinal Encoding ──────────────────────────────────────────────────────
# Best for: ordered categories — MUST define order yourself

edu_df = demo_df[['education']].fillna('High School').copy()

# Define the explicit order
edu_order = [['High School', 'Bachelor', 'Master', 'PhD']]

oe = OrdinalEncoder(categories=edu_order)
edu_df['education_encoded'] = oe.fit_transform(edu_df[['education']])

print('Education Ordinal Mapping:')
for i, cat in enumerate(edu_order[0]):
    print(f'  {cat:12s} → {i}')

print('\nSample rows:')
print(edu_df.sample(8, random_state=1).to_string(index=False))

# Alternative — manual dict map (even clearer)
edu_map = {'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3}
edu_df['edu_manual'] = edu_df['education'].map(edu_map)
print('\n✅ Ordinal encoding preserves ordering. PhD(3) > Master(2) > Bachelor(1) > HS(0)')

In [ ]:
# ── Label Encoding ────────────────────────────────────────────────────────
# Use ONLY with tree-based models or binary columns
# NEVER with linear models on nominal data (implies false ordering)

le = LabelEncoder()
dept_label = le.fit_transform(demo_df['department'])

print('Label Encoding — Department:')
for cls, code in zip(le.classes_, range(len(le.classes_))):
    print(f'  {cls:12s} → {code}')

print('\n⚠️  DANGER: Engineering=0 < Finance=1 < HR=2 < Marketing=3 < Sales=4')
print('   Linear model would think HR is "between" Finance and Marketing!')
print('   Safe only with Decision Trees / Random Forests.')

In [ ]:
# ── Target Encoding ───────────────────────────────────────────────────────
# Replaces category with MEAN of target — great for high cardinality
# Must be done inside cross-validation to prevent data leakage!

df_te = demo_df[['country', 'churned']].copy()

# Naive (leaky) version — for illustration only
target_mean = df_te.groupby('country')['churned'].mean()
df_te['country_target_naive'] = df_te['country'].map(target_mean)

print('Target Encoding — Country → Churn Rate:')
print(target_mean.sort_values(ascending=False).to_string())

# Proper version with smoothing (blends category mean with global mean)
global_mean = df_te['churned'].mean()
smoothing = 10  # higher = more shrinkage toward global mean

def smooth_target_encode(series, target, smoothing=10):
    stats_ = target.groupby(series).agg(['mean', 'count'])
    smoother = 1 / (1 + np.exp(-(stats_['count'] - smoothing)))
    encoded = smoother * stats_['mean'] + (1 - smoother) * target.mean()
    return series.map(encoded)

df_te['country_target_smooth'] = smooth_target_encode(
    df_te['country'], df_te['churned'], smoothing=10
)

print(f'\nGlobal churn rate: {global_mean:.3f}')
print('\nNaive vs Smoothed encoding:')
compare = pd.DataFrame({
    'naive':    df_te.groupby('country')['country_target_naive'].first(),
    'smoothed': df_te.groupby('country')['country_target_smooth'].first()
})
print(compare.round(3))

In [ ]:
# ── Frequency & Binary Encoding ───────────────────────────────────────────

# Frequency encoding — replace with count or proportion
freq_map = demo_df['country'].value_counts(normalize=True)  # proportions
demo_df['country_freq_enc'] = demo_df['country'].map(freq_map)

print('Frequency Encoding — Country:')
print(freq_map.to_string())

# Binary encoding (category_encoders)
be = BinaryEncoder(cols=['department'], drop_invariant=True)
dept_binary = be.fit_transform(demo_df[['department']])

print(f'\nBinary Encoding: {demo_df["department"].nunique()} categories'
      f' → {dept_binary.shape[1]} columns')
print('(vs One-Hot which would need', demo_df['department'].nunique() - 1, 'columns)')
print(dept_binary.head())

# Summary of all encoding methods
print('\n' + '='*60)
print('ENCODING SUMMARY')
print('='*60)
methods = [
    ('One-Hot',   'Low cardinality nominal',  'Linear models, NN',  'Memory'),
    ('Ordinal',   'Ordered categories',        'All models',         'Must know order'),
    ('Label',     'Binary / Tree models only', 'Tree models',        'False order for linear'),
    ('Target',    'High cardinality',          'Trees especially',   'Leakage risk'),
    ('Frequency', 'Any cardinality',           'Quick baseline',     'Collisions'),
    ('Binary',    'Medium-high cardinality',   'Trees',              'Harder to interpret'),
]
for m in methods:
    print(f'  {m[0]:12s} | Use: {m[1]:30s} | Best for: {m[2]:25s} | Risk: {m[3]}')

---
# 📕 PART 3: Scaling, Normalization & Transformations
---

> Distance-based and gradient-descent-based models break when features have different scales.
> Tree-based models are scale-invariant — they don't need scaling.

In [ ]:
# ── Compare All Scalers Side-by-Side ─────────────────────────────────────
# Use salary (right-skewed, has outliers) as example

salary_arr = df_work['salary_median_imputed'].values.reshape(-1, 1)

scalers = {
    'Original':            salary_arr,
    'StandardScaler\n(z = (x-μ)/σ)':    StandardScaler().fit_transform(salary_arr),
    'MinMaxScaler\n[0, 1]':              MinMaxScaler().fit_transform(salary_arr),
    'RobustScaler\n(median/IQR)':        RobustScaler().fit_transform(salary_arr),
    'PowerTransformer\n(Yeo-Johnson)':   PowerTransformer(method='yeo-johnson').fit_transform(salary_arr),
    'QuantileTransformer\n→ Normal':     QuantileTransformer(output_distribution='normal',
                                                               n_quantiles=100, random_state=42).fit_transform(salary_arr),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Effect of Different Scaling Methods on Salary', fontsize=14, fontweight='bold')

for ax, (name, data) in zip(axes.flatten(), scalers.items()):
    ax.hist(data.flatten(), bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_title(name, fontsize=10)
    mu, sigma = data.mean(), data.std()
    ax.set_xlabel(f'μ={mu:.2f}, σ={sigma:.2f}', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Log Transform Deep Dive ───────────────────────────────────────────────
# Why: right-skewed data (salary, house price, revenue) violates normality assumptions

salary_raw = df_work['salary_median_imputed']
salary_log = np.log1p(salary_raw)  # log(1+x) — safe for 0 values
salary_sqrt = np.sqrt(salary_raw)  # milder transformation

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Distribution Transformations — Salary', fontsize=13, fontweight='bold')

for col_idx, (data, label) in enumerate([
    (salary_raw, 'Original'),
    (salary_log, 'Log Transform'),
    (salary_sqrt, 'Square Root Transform')
]):
    # Histogram
    axes[0, col_idx].hist(data, bins=30, color='steelblue', edgecolor='white')
    sk = skew(data)
    axes[0, col_idx].set_title(f'{label}\nskewness = {sk:.2f}')

    # Q-Q plot
    probplot(data, plot=axes[1, col_idx])
    axes[1, col_idx].set_title(f'{label} — Q-Q Plot')

plt.tight_layout()
plt.show()

print('Skewness comparison:')
print(f'  Original   : {skew(salary_raw):.3f}')
print(f'  Log(1+x)   : {skew(salary_log):.3f}  ← closest to 0 is best')
print(f'  Square root: {skew(salary_sqrt):.3f}')

In [ ]:
# ── The Golden Rule: ALWAYS Fit on Train, Transform Both ─────────────────
print('⚠️  THE MOST IMPORTANT RULE IN PREPROCESSING:')
print('   Fit your scaler ONLY on training data.')
print('   Apply (transform) to both train and test.')
print()

# Toy example demonstrating leakage vs correct approach
X_dummy = pd.DataFrame({'salary': salary_raw.values, 'years_exp': demo_df['years_exp'].values})
y_dummy = demo_df['churned'].values

X_tr, X_te, y_tr, y_te = train_test_split(X_dummy, y_dummy, test_size=0.2, random_state=42)

# ✅ CORRECT
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)   # fit AND transform on train
X_te_scaled = scaler.transform(X_te)        # ONLY transform on test

print('✅ CORRECT:')
print(f'   Train salary mean used for scaling: {scaler.mean_[0]:.2f}')
print(f'   Test is scaled using TRAIN mean — test data never seen during fit')

# ❌ WRONG (leakage)
scaler_bad = StandardScaler()
X_te_leaky = scaler_bad.fit_transform(X_te)  # refits on test!

print('\n❌ LEAKY:')
print(f'   Test mean used for scaling: {scaler_bad.mean_[0]:.2f}')
print(f'   This is different from train mean! Model saw test info during training.')

print('\n💡 Solution: always use sklearn Pipeline — it handles this automatically!')

---
# 📗 PART 4: Feature Creation
---

> The most creative part of FE. Raw columns rarely tell the full story — new features do.

In [ ]:
# ── Mathematical Combinations ─────────────────────────────────────────────
fe_df = demo_df.copy()
fe_df['salary'] = fe_df['salary'].fillna(fe_df['salary'].median())

# Ratios — capture relationships better than raw values
fe_df['salary_per_year_exp']  = fe_df['salary'] / (fe_df['years_exp'] + 1)
fe_df['bmi']                  = fe_df['weight_kg'] / ((fe_df['height_cm'] / 100) ** 2)

# Differences
mean_salary = fe_df['salary'].mean()
fe_df['salary_above_mean']    = fe_df['salary'] - mean_salary

# Log transforms (for skewed features to use in models)
fe_df['log_salary']           = np.log1p(fe_df['salary'])

# Polynomial — square and cube of numeric features
fe_df['years_exp_sq']         = fe_df['years_exp'] ** 2
fe_df['years_exp_cu']         = fe_df['years_exp'] ** 3

# Interaction: multiply two features
fe_df['salary_x_exp']         = fe_df['salary'] * fe_df['years_exp']

print('New math-based features created:')
new_cols = ['salary_per_year_exp', 'bmi', 'salary_above_mean',
            'log_salary', 'years_exp_sq', 'salary_x_exp']
print(fe_df[new_cols].describe().round(2))

In [ ]:
# ── DateTime Feature Engineering ──────────────────────────────────────────
print('🗓️  DateTime Feature Engineering')
print('='*50)

fe_df['hire_date'] = pd.to_datetime(fe_df['hire_date'])
today = pd.Timestamp('2024-12-01')

# Calendar components
fe_df['hire_year']       = fe_df['hire_date'].dt.year
fe_df['hire_month']      = fe_df['hire_date'].dt.month
fe_df['hire_day']        = fe_df['hire_date'].dt.day
fe_df['hire_dayofweek']  = fe_df['hire_date'].dt.dayofweek    # 0=Mon
fe_df['hire_quarter']    = fe_df['hire_date'].dt.quarter

# Boolean flags
fe_df['hired_on_weekend']   = (fe_df['hire_date'].dt.dayofweek >= 5).astype(int)
fe_df['hired_in_q4']        = (fe_df['hire_date'].dt.quarter == 4).astype(int)

# Duration features
fe_df['tenure_days']   = (today - fe_df['hire_date']).dt.days
fe_df['tenure_years']  = fe_df['tenure_days'] / 365.25

# ── Cyclical Encoding — CRITICAL for periodic features ─────────────────
# Problem: December (12) and January (1) are close but numerically far
# Solution: encode as sin + cos (2D circle)
fe_df['month_sin']     = np.sin(2 * np.pi * fe_df['hire_month'] / 12)
fe_df['month_cos']     = np.cos(2 * np.pi * fe_df['hire_month'] / 12)
fe_df['dow_sin']       = np.sin(2 * np.pi * fe_df['hire_dayofweek'] / 7)
fe_df['dow_cos']       = np.cos(2 * np.pi * fe_df['hire_dayofweek'] / 7)

# Visualise cyclical encoding
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Cyclical Encoding of Month', fontsize=13, fontweight='bold')

months = np.arange(1, 13)
sin_vals = np.sin(2 * np.pi * months / 12)
cos_vals = np.cos(2 * np.pi * months / 12)

axes[0].plot(months, sin_vals, 'o-', color='steelblue', label='sin(month)')
axes[0].plot(months, cos_vals, 's-', color='coral', label='cos(month)')
axes[0].set_xlabel('Month')
axes[0].set_title('Sin/Cos Components')
axes[0].legend()
axes[0].set_xticks(months)

axes[1].scatter(sin_vals, cos_vals, c=months, cmap='hsv', s=100, zorder=5)
for i, m in enumerate(months):
    axes[1].annotate(str(m), (sin_vals[i], cos_vals[i]), fontsize=9, ha='center')
axes[1].set_title('Month on Unit Circle\n(Dec & Jan are now CLOSE)')
axes[1].set_xlabel('sin(month)')
axes[1].set_ylabel('cos(month)')

plt.tight_layout()
plt.show()

print(f'\n✅ DateTime features created:')
dt_features = ['tenure_days', 'tenure_years', 'hire_quarter', 'month_sin', 'month_cos']
print(fe_df[dt_features].describe().round(3))

In [ ]:
# ── Binning / Discretisation ──────────────────────────────────────────────
print('📊 Binning: Converting continuous → categories')

# Equal-width bins
fe_df['age_group'] = pd.cut(
    fe_df['years_exp'],
    bins=[0, 5, 10, 15, 20, 30],
    labels=['Junior', 'Mid', 'Senior', 'Lead', 'Principal'],
    include_lowest=True
)

# Equal-frequency bins (quantile-based)
fe_df['salary_quartile'] = pd.qcut(
    fe_df['salary'],
    q=4,
    labels=['Q1-Low', 'Q2-MidLow', 'Q3-MidHigh', 'Q4-High']
)

# Custom bins based on domain knowledge
fe_df['bmi_category'] = pd.cut(
    fe_df['bmi'],
    bins=[0, 18.5, 25, 30, 100],
    labels=['Underweight', 'Normal', 'Overweight', 'Obese']
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Binning Examples', fontsize=13, fontweight='bold')

for ax, col, title in zip(axes,
    ['age_group', 'salary_quartile', 'bmi_category'],
    ['Experience → Seniority', 'Salary → Quartile', 'BMI → Category']):
    fe_df[col].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# ── Group Aggregation Features ────────────────────────────────────────────
# Compute group-level statistics and attach them back to each row

# Department-level salary stats (attach back to each employee)
dept_salary_stats = fe_df.groupby('department')['salary'].agg(
    dept_salary_mean='mean',
    dept_salary_median='median',
    dept_salary_std='std',
    dept_salary_max='max',
    dept_headcount='count'
).reset_index()

fe_df = fe_df.merge(dept_salary_stats, on='department', how='left')

# Relative salary: how much above/below dept average?
fe_df['salary_vs_dept_mean'] = fe_df['salary'] - fe_df['dept_salary_mean']
fe_df['salary_pct_of_dept']  = fe_df['salary'] / fe_df['dept_salary_mean']

print('Group Aggregation Features (Department-level stats per employee):')
cols = ['department', 'salary', 'dept_salary_mean', 'salary_vs_dept_mean', 'salary_pct_of_dept']
print(fe_df[cols].drop_duplicates('department').sort_values('department').to_string(index=False))

---
# 📘 PART 5: Feature Selection
---

> More features ≠ better model. Irrelevant features add noise and slow training.

In [ ]:
# ── Step 1: Remove Zero/Near-Zero Variance Features ───────────────────────
from sklearn.feature_selection import VarianceThreshold

# Build a numeric feature matrix for selection demo
selection_df = fe_df.select_dtypes(include='number').drop(
    columns=['churned'], errors='ignore'
).dropna()

# Add an artificial constant column (should be removed)
selection_df['constant_col'] = 1.0
selection_df['near_constant'] = np.random.choice([0, 1], len(selection_df), p=[0.999, 0.001])

print(f'Features before variance filter: {selection_df.shape[1]}')

vt = VarianceThreshold(threshold=0.01)
vt.fit(selection_df)

removed = selection_df.columns[~vt.get_support()].tolist()
kept    = selection_df.columns[vt.get_support()].tolist()

print(f'Features after variance filter:  {sum(vt.get_support())}')
print(f'Removed: {removed}')

# Show variances
var_df = pd.DataFrame({'Feature': selection_df.columns, 'Variance': vt.variances_})
print('\nBottom 5 by variance:')
print(var_df.sort_values('Variance').head(5).to_string(index=False))

In [ ]:
# ── Step 2: Remove Highly Correlated Features ─────────────────────────────
def remove_high_corr(df, threshold=0.90):
    """Drop one of each pair with correlation above threshold."""
    corr_matrix = df.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    return df.drop(columns=to_drop), to_drop

selection_clean = selection_df[kept].copy()
reduced_df, dropped_corr = remove_high_corr(selection_clean, threshold=0.90)

print(f'Features before correlation filter: {selection_clean.shape[1]}')
print(f'Features after  correlation filter: {reduced_df.shape[1]}')
print(f'Dropped (high correlation): {dropped_corr}')

In [ ]:
# ── Step 3: Univariate Statistical Feature Selection ──────────────────────

y_sel = fe_df.loc[reduced_df.index, 'churned'].values
X_sel = reduced_df.copy()

# F-statistic (ANOVA) — measures linear relationship
selector_f  = SelectKBest(score_func=f_classif, k='all')
selector_f.fit(X_sel, y_sel)

# Mutual Information — captures non-linear relationships
selector_mi = SelectKBest(score_func=mutual_info_classif, k='all')
selector_mi.fit(X_sel, y_sel)

scores_df = pd.DataFrame({
    'Feature': X_sel.columns,
    'F_score': selector_f.scores_,
    'F_pvalue': selector_f.pvalues_,
    'MI_score': selector_mi.scores_
}).sort_values('MI_score', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Univariate Feature Importance', fontsize=13, fontweight='bold')

top15 = scores_df.head(15)

top15.sort_values('F_score').plot(kind='barh', x='Feature', y='F_score',
                                   ax=axes[0], color='steelblue', legend=False)
axes[0].set_title('F-Score (ANOVA)')

top15.sort_values('MI_score').plot(kind='barh', x='Feature', y='MI_score',
                                    ax=axes[1], color='coral', legend=False)
axes[1].set_title('Mutual Information Score')

plt.tight_layout()
plt.show()

# Features significant at p < 0.05
sig_features = scores_df[scores_df['F_pvalue'] < 0.05]['Feature'].tolist()
print(f'\n✅ Features significant at p<0.05 (F-test): {sig_features}')

In [ ]:
# ── Step 4: Tree-Based Feature Importance ─────────────────────────────────
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_sel, y_sel, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_tr2, y_tr2)

imp_df = pd.DataFrame({
    'Feature': X_sel.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot
plt.figure(figsize=(10, 7))
imp_df.head(15).sort_values('Importance').plot(
    kind='barh', x='Feature', y='Importance',
    color='steelblue', legend=False
)
plt.title('Random Forest Feature Importances (Top 15)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

# Keep top N features
top_n = 12
top_features = imp_df.head(top_n)['Feature'].tolist()
print(f'\nTop {top_n} features by Random Forest importance:')
for i, (_, row) in enumerate(imp_df.head(top_n).iterrows(), 1):
    print(f'  {i:2d}. {row["Feature"]:30s} {row["Importance"]:.4f}')

In [ ]:
# ── Step 5: RFE + Lasso Embedded Selection ────────────────────────────────
from sklearn.preprocessing import StandardScaler

scaler_sel = StandardScaler()
X_scaled = scaler_sel.fit_transform(X_tr2)
X_te_scaled = scaler_sel.transform(X_te2)

# RFE with Logistic Regression
rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=8)
rfe.fit(X_scaled, y_tr2)

rfe_features = X_sel.columns[rfe.support_].tolist()
print(f'RFE selected {len(rfe_features)} features:')
print(rfe_features)

# Lasso embedded selection
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=5000)
lasso_cv.fit(X_scaled, y_tr2)

lasso_selector = SelectFromModel(lasso_cv, prefit=True)
lasso_features = X_sel.columns[lasso_selector.get_support()].tolist()

print(f'\nLasso (alpha={lasso_cv.alpha_:.4f}) selected {len(lasso_features)} features:')
print(lasso_features)

# Union of agreed features
agreed = list(set(rfe_features) & set(lasso_features))
print(f'\n✅ Features agreed by both RFE and Lasso: {agreed}')

---
# 🔥 PART 6: FULL END-TO-END PROJECT
## Customer Churn Prediction — Telecom Dataset
---

**Scenario:** You are a Data Scientist at a telecom company. Marketing wants to identify customers likely to churn in the next month so they can proactively offer discounts.

**Goal:** Build the best possible churn prediction model using everything in Parts 1–5.

**Steps:**
1. Generate realistic synthetic dataset
2. Full EDA
3. Feature engineering pipeline
4. Model comparison
5. Final model evaluation & insights

In [ ]:
# ════════════════════════════════════════════════════════════════
# 6.1  GENERATE REALISTIC TELECOM CHURN DATASET
# ════════════════════════════════════════════════════════════════
np.random.seed(2024)
N = 5000

def generate_churn_dataset(n=5000, seed=2024):
    """Generate a realistic synthetic telecom churn dataset."""
    rng = np.random.default_rng(seed)

    # Demographics
    age          = rng.integers(18, 80, n)
    gender       = rng.choice(['Male', 'Female'], n)
    senior       = (age >= 65).astype(int)
    state        = rng.choice(['CA', 'TX', 'NY', 'FL', 'WA', 'IL', 'OH', 'PA', 'AZ', 'GA'], n)

    # Account info
    tenure_months = rng.integers(1, 72, n)
    contract      = rng.choice(['Month-to-month', 'One year', 'Two year'], n,
                                p=[0.55, 0.25, 0.20])
    paperless     = rng.choice([0, 1], n, p=[0.4, 0.6])
    payment_method = rng.choice(['Electronic check', 'Mailed check',
                                  'Bank transfer', 'Credit card'], n,
                                  p=[0.34, 0.23, 0.22, 0.21])

    # Services
    phone_service   = rng.choice([0, 1], n, p=[0.1, 0.9])
    multiple_lines  = rng.choice([0, 1], n, p=[0.5, 0.5]) * phone_service
    internet_service = rng.choice(['None', 'DSL', 'Fiber optic'], n, p=[0.21, 0.34, 0.45])
    online_security  = rng.choice([0, 1], n) * (internet_service != 'None')
    tech_support     = rng.choice([0, 1], n) * (internet_service != 'None')
    streaming_tv     = rng.choice([0, 1], n) * (internet_service != 'None')
    streaming_movies = rng.choice([0, 1], n) * (internet_service != 'None')

    # Charges (realistic pricing)
    base_monthly = (
        20 * phone_service +
        15 * (internet_service == 'DSL') +
        30 * (internet_service == 'Fiber optic') +
        5 * multiple_lines + 5 * online_security +
        5 * tech_support + 8 * streaming_tv + 8 * streaming_movies
    )
    monthly_charges = base_monthly + rng.normal(0, 5, n)
    monthly_charges = np.clip(monthly_charges, 10, 120)
    total_charges   = monthly_charges * tenure_months + rng.normal(0, 50, n)
    total_charges   = np.clip(total_charges, 0, None)

    # Support calls (higher = more frustrated)
    support_calls = rng.poisson(1.5, n)

    # Customer satisfaction (1-5)
    satisfaction = rng.choice([1, 2, 3, 4, 5], n, p=[0.1, 0.2, 0.3, 0.25, 0.15])

    # Churn probability (realistic — depends on multiple factors)
    churn_prob = (
        0.05
        + 0.30 * (contract == 'Month-to-month')
        - 0.10 * (contract == 'Two year')
        + 0.15 * (internet_service == 'Fiber optic')   # Fiber has more complaints
        - 0.05 * online_security
        - 0.05 * tech_support
        + 0.02 * support_calls
        - 0.04 * (satisfaction >= 4)
        + 0.08 * (satisfaction <= 2)
        + 0.10 * (payment_method == 'Electronic check')
        - 0.003 * tenure_months   # long-term customers less likely to churn
        + rng.normal(0, 0.05, n)
    )
    churn_prob = np.clip(churn_prob, 0.01, 0.99)
    churn      = (rng.random(n) < churn_prob).astype(int)

    # Inject missing values (~3%)
    for col_idx in rng.choice(n, int(n * 0.03), replace=False):
        total_charges[col_idx] = np.nan

    df = pd.DataFrame({
        'customer_id':       [f'CUST_{i:05d}' for i in range(n)],
        'age':               age,
        'gender':            gender,
        'senior_citizen':    senior,
        'state':             state,
        'tenure_months':     tenure_months,
        'contract':          contract,
        'paperless_billing': paperless,
        'payment_method':    payment_method,
        'phone_service':     phone_service,
        'multiple_lines':    multiple_lines,
        'internet_service':  internet_service,
        'online_security':   online_security,
        'tech_support':      tech_support,
        'streaming_tv':      streaming_tv,
        'streaming_movies':  streaming_movies,
        'monthly_charges':   monthly_charges.round(2),
        'total_charges':     total_charges.round(2),
        'support_calls':     support_calls,
        'satisfaction_score': satisfaction,
        'churn':             churn
    })
    return df

raw_df = generate_churn_dataset(N)
print(f'✅ Dataset created: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns')
print(f'   Churn rate: {raw_df["churn"].mean():.1%}')
raw_df.head()

In [ ]:
# ════════════════════════════════════════════════════════════════
# 6.2  PROJECT EDA
# ════════════════════════════════════════════════════════════════

print('📊 Dataset Overview')
print(raw_df.info())
print('\n📋 Missing Values:')
print(raw_df.isnull().sum()[raw_df.isnull().sum() > 0])
print(f'\n🎯 Target Distribution:')
print(raw_df['churn'].value_counts())
print(raw_df['churn'].value_counts(normalize=True).round(3))

In [ ]:
# ── Comprehensive EDA Dashboard ───────────────────────────────────────────
fig = plt.figure(figsize=(20, 16))
fig.suptitle('🔍 Telecom Churn — EDA Dashboard', fontsize=16, fontweight='bold', y=1.01)

gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.5, wspace=0.4)

# 1. Churn rate (pie)
ax1 = fig.add_subplot(gs[0, 0])
churn_counts = raw_df['churn'].value_counts()
ax1.pie(churn_counts, labels=['No Churn', 'Churn'], autopct='%1.1f%%',
        colors=['steelblue', 'tomato'], startangle=90)
ax1.set_title('Churn Distribution')

# 2. Churn rate by contract
ax2 = fig.add_subplot(gs[0, 1])
churn_by_contract = raw_df.groupby('contract')['churn'].mean().sort_values(ascending=False)
churn_by_contract.plot(kind='bar', ax=ax2, color=['tomato','orange','green'], edgecolor='white')
ax2.set_title('Churn Rate by Contract')
ax2.set_ylabel('Churn Rate')
ax2.tick_params(axis='x', rotation=15)
ax2.axhline(raw_df['churn'].mean(), color='navy', linestyle='--', alpha=0.7)

# 3. Churn rate by internet service
ax3 = fig.add_subplot(gs[0, 2])
churn_by_internet = raw_df.groupby('internet_service')['churn'].mean().sort_values(ascending=False)
churn_by_internet.plot(kind='bar', ax=ax3, color='coral', edgecolor='white')
ax3.set_title('Churn Rate by Internet Service')
ax3.set_ylabel('Churn Rate')
ax3.tick_params(axis='x', rotation=15)

# 4. Monthly charges distribution
ax4 = fig.add_subplot(gs[0, 3])
for label, grp in raw_df.groupby('churn')['monthly_charges']:
    ax4.hist(grp, bins=30, alpha=0.6, label=f'Churn={label}', edgecolor='white')
ax4.set_title('Monthly Charges by Churn')
ax4.legend()

# 5. Tenure distribution
ax5 = fig.add_subplot(gs[1, 0])
for label, grp in raw_df.groupby('churn')['tenure_months']:
    ax5.hist(grp, bins=20, alpha=0.6, label=f'Churn={label}', edgecolor='white')
ax5.set_title('Tenure by Churn')
ax5.legend()

# 6. Satisfaction score
ax6 = fig.add_subplot(gs[1, 1])
sat_churn = raw_df.groupby('satisfaction_score')['churn'].mean()
sat_churn.plot(kind='bar', ax=ax6, color='steelblue', edgecolor='white')
ax6.set_title('Churn Rate by Satisfaction Score')
ax6.set_xlabel('Satisfaction (1=Low, 5=High)')
ax6.set_ylabel('Churn Rate')
ax6.tick_params(axis='x', rotation=0)

# 7. Support calls
ax7 = fig.add_subplot(gs[1, 2])
call_churn = raw_df.groupby('support_calls')['churn'].mean()
call_churn.plot(kind='bar', ax=ax7, color='salmon', edgecolor='white')
ax7.set_title('Churn Rate by Support Calls')
ax7.set_xlabel('Number of Support Calls')
ax7.tick_params(axis='x', rotation=0)

# 8. Correlation heatmap
ax8 = fig.add_subplot(gs[1, 3])
numeric_cols_proj = ['age', 'tenure_months', 'monthly_charges', 'support_calls', 'satisfaction_score', 'churn']
corr_proj = raw_df[numeric_cols_proj].corr()
sns.heatmap(corr_proj, annot=True, fmt='.2f', cmap='RdBu_r', ax=ax8,
            center=0, vmin=-1, vmax=1, square=False, linewidths=0.3, cbar=False)
ax8.set_title('Correlation Matrix')

# 9. Age distribution
ax9 = fig.add_subplot(gs[2, 0])
for label, grp in raw_df.groupby('churn')['age']:
    ax9.hist(grp, bins=20, alpha=0.6, label=f'Churn={label}', edgecolor='white')
ax9.set_title('Age Distribution by Churn')
ax9.legend()

# 10. Payment method
ax10 = fig.add_subplot(gs[2, 1])
pay_churn = raw_df.groupby('payment_method')['churn'].mean().sort_values(ascending=False)
pay_churn.plot(kind='bar', ax=ax10, color='orchid', edgecolor='white')
ax10.set_title('Churn Rate by Payment Method')
ax10.tick_params(axis='x', rotation=20)

# 11. Gender
ax11 = fig.add_subplot(gs[2, 2])
gender_churn = raw_df.groupby('gender')['churn'].mean()
gender_churn.plot(kind='bar', ax=ax11, color=['steelblue','coral'], edgecolor='white')
ax11.set_title('Churn Rate by Gender')
ax11.tick_params(axis='x', rotation=0)

# 12. State churn rate
ax12 = fig.add_subplot(gs[2, 3])
state_churn = raw_df.groupby('state')['churn'].mean().sort_values(ascending=False)
state_churn.plot(kind='bar', ax=ax12, color='teal', edgecolor='white')
ax12.set_title('Churn Rate by State')
ax12.tick_params(axis='x', rotation=30)

plt.savefig('eda_dashboard.png', bbox_inches='tight', dpi=100)
plt.show()
print('✅ EDA Dashboard complete!')

In [ ]:
# ════════════════════════════════════════════════════════════════
# 6.3  FULL FEATURE ENGINEERING PIPELINE
# ════════════════════════════════════════════════════════════════

def engineer_features(df):
    """
    Complete feature engineering pipeline for churn prediction.
    Input : raw dataframe
    Output: feature-engineered dataframe ready for ML
    """
    df = df.copy()

    # ── 1. HANDLE MISSING VALUES ───────────────────────────────────────────
    # Add missing indicator before imputing
    df['total_charges_missing'] = df['total_charges'].isnull().astype(int)
    # Impute with median (robust to outliers)
    df['total_charges'] = df['total_charges'].fillna(df['total_charges'].median())

    # ── 2. NEW FEATURES — DOMAIN KNOWLEDGE ────────────────────────────────

    # Charge-based features
    df['avg_monthly_charge_per_service'] = df['monthly_charges'] / (
        df[['phone_service','multiple_lines','online_security',
            'tech_support','streaming_tv','streaming_movies']].sum(axis=1) + 1
    )
    df['total_vs_expected'] = df['total_charges'] / (df['monthly_charges'] * df['tenure_months'] + 1)
    df['monthly_charge_log'] = np.log1p(df['monthly_charges'])
    df['total_charge_log']   = np.log1p(df['total_charges'])
    df['charges_ratio']      = df['monthly_charges'] / (df['total_charges'] + 1)

    # Tenure-based features
    df['is_new_customer']    = (df['tenure_months'] <= 3).astype(int)
    df['is_loyal_customer']  = (df['tenure_months'] >= 36).astype(int)
    df['tenure_log']         = np.log1p(df['tenure_months'])
    df['tenure_sq']          = df['tenure_months'] ** 2
    df['tenure_band']        = pd.cut(df['tenure_months'],
                                       bins=[0, 6, 12, 24, 48, 72],
                                       labels=[0, 1, 2, 3, 4]).astype(float)

    # Service count features
    service_cols = ['phone_service','multiple_lines','online_security',
                    'tech_support','streaming_tv','streaming_movies']
    df['total_services']   = df[service_cols].sum(axis=1)
    df['has_no_services']  = (df['total_services'] == 0).astype(int)
    df['has_all_streaming']= (df['streaming_tv'] & df['streaming_movies']).astype(int)
    df['has_protection']   = (df['online_security'] | df['tech_support']).astype(int)

    # Age features
    df['age_group'] = pd.cut(df['age'],
                              bins=[17, 30, 45, 60, 80],
                              labels=['Young', 'MiddleAge', 'Senior', 'Elder']
                             ).astype(str)
    df['is_young']  = (df['age'] < 30).astype(int)
    df['is_elder']  = (df['age'] > 60).astype(int)

    # Support call risk
    df['high_support_usage'] = (df['support_calls'] >= 3).astype(int)
    df['support_x_satisfaction'] = df['support_calls'] * (6 - df['satisfaction_score'])

    # Satisfaction features
    df['is_dissatisfied']  = (df['satisfaction_score'] <= 2).astype(int)
    df['is_satisfied']     = (df['satisfaction_score'] >= 4).astype(int)

    # Interaction features
    df['fiber_month_to_month'] = (
        (df['internet_service'] == 'Fiber optic') &
        (df['contract'] == 'Month-to-month')
    ).astype(int)

    df['echeque_no_security'] = (
        (df['payment_method'] == 'Electronic check') &
        (df['online_security'] == 0)
    ).astype(int)

    df['senior_no_support'] = (
        (df['senior_citizen'] == 1) &
        (df['tech_support'] == 0)
    ).astype(int)

    df['new_and_expensive'] = (
        (df['tenure_months'] <= 6) &
        (df['monthly_charges'] > df['monthly_charges'].quantile(0.75))
    ).astype(int)

    # State-level aggregation features
    state_stats = df.groupby('state').agg(
        state_avg_charges=('monthly_charges', 'mean'),
        state_avg_tenure=('tenure_months', 'mean')
    ).reset_index()
    df = df.merge(state_stats, on='state', how='left')

    # ── 3. ENCODE CATEGORICALS ─────────────────────────────────────────────

    # Binary: gender
    df['gender_encoded'] = (df['gender'] == 'Male').astype(int)

    # Ordinal: contract (longer contract = lower churn risk)
    contract_map = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
    df['contract_encoded'] = df['contract'].map(contract_map)

    # One-hot: internet_service (3 levels, low cardinality)
    df['internet_dsl']   = (df['internet_service'] == 'DSL').astype(int)
    df['internet_fiber'] = (df['internet_service'] == 'Fiber optic').astype(int)
    # 'None' = baseline (both 0)

    # One-hot: payment_method
    df['pay_echeque']   = (df['payment_method'] == 'Electronic check').astype(int)
    df['pay_mcheque']   = (df['payment_method'] == 'Mailed check').astype(int)
    df['pay_bank']      = (df['payment_method'] == 'Bank transfer').astype(int)
    # 'Credit card' = baseline

    # Frequency: state (high cardinality)
    state_freq = df['state'].value_counts(normalize=True)
    df['state_freq'] = df['state'].map(state_freq)

    # Ordinal: age_group
    age_group_map = {'Young': 0, 'MiddleAge': 1, 'Senior': 2, 'Elder': 3}
    df['age_group_enc'] = df['age_group'].map(age_group_map)

    return df

engineered_df = engineer_features(raw_df)

print(f'✅ Feature engineering complete!')
print(f'   Original features : {raw_df.shape[1]}')
print(f'   After engineering : {engineered_df.shape[1]}')
print(f'   New features added: {engineered_df.shape[1] - raw_df.shape[1]}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# 6.4  PREPARE TRAIN/TEST SPLIT
# ════════════════════════════════════════════════════════════════

# Columns to drop from model input
drop_cols = [
    'customer_id', 'churn',
    # Raw categoricals (already encoded)
    'gender', 'contract', 'internet_service', 'payment_method',
    'state', 'age_group'
]

# Select only numeric columns (all encoding is done)
feature_cols = [c for c in engineered_df.select_dtypes(include='number').columns
                if c not in drop_cols and c != 'churn']

X = engineered_df[feature_cols]
y = engineered_df['churn']

print(f'Feature matrix: {X.shape[0]:,} rows × {X.shape[1]} features')
print(f'Target: {y.sum():,} churned / {(1-y).sum():,} retained ({y.mean():.1%} churn rate)')

# Stratified split (preserve class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'\nTrain set: {X_train.shape[0]:,} rows')
print(f'Test set : {X_test.shape[0]:,} rows')
print(f'Train churn: {y_train.mean():.1%} | Test churn: {y_test.mean():.1%}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# 6.5  MODEL COMPARISON — 5 ALGORITHMS
# ════════════════════════════════════════════════════════════════

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=42, C=1.0))
    ]),
    'Random Forest': Pipeline([
        ('model', RandomForestClassifier(n_estimators=200, max_depth=8,
                                          random_state=42, n_jobs=-1))
    ]),
    'Gradient Boosting': Pipeline([
        ('model', GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                              learning_rate=0.05, random_state=42))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=11, weights='distance'))
    ]),
    'XGBoost': Pipeline([
        ('model', XGBClassifier(n_estimators=200, max_depth=4,
                                 learning_rate=0.05, use_label_encoder=False,
                                 eval_metric='logloss', random_state=42,
                                 verbosity=0, n_jobs=-1))
    ])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print('Training and cross-validating models...')
print('-' * 70)
print(f'{"Model":<25} {"CV AUC":>10} {"CV F1":>10} {"CV Acc":>10}')
print('-' * 70)

for name, pipeline in models.items():
    auc_scores = cross_val_score(pipeline, X_train, y_train, cv=cv,
                                  scoring='roc_auc', n_jobs=-1)
    f1_scores  = cross_val_score(pipeline, X_train, y_train, cv=cv,
                                  scoring='f1', n_jobs=-1)
    acc_scores = cross_val_score(pipeline, X_train, y_train, cv=cv,
                                  scoring='accuracy', n_jobs=-1)

    results[name] = {
        'auc_mean': auc_scores.mean(), 'auc_std': auc_scores.std(),
        'f1_mean': f1_scores.mean(),   'f1_std': f1_scores.std(),
        'acc_mean': acc_scores.mean(), 'acc_std': acc_scores.std()
    }

    print(f'{name:<25} {auc_scores.mean():.4f}±{auc_scores.std():.3f} '
          f'{f1_scores.mean():.4f}±{f1_scores.std():.3f} '
          f'{acc_scores.mean():.4f}±{acc_scores.std():.3f}')

print('-' * 70)

In [ ]:
# ── Model Comparison Visualisation ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Comparison — 5-Fold Cross-Validation', fontsize=14, fontweight='bold')

metrics = ['auc_mean', 'f1_mean', 'acc_mean']
metric_labels = ['ROC-AUC', 'F1 Score', 'Accuracy']
colors = ['steelblue', 'coral', 'green', 'orchid', 'goldenrod']

for ax, metric, label in zip(axes, metrics, metric_labels):
    model_names = list(results.keys())
    values = [results[m][metric] for m in model_names]
    std_vals = [results[m][metric.replace('mean','std')] for m in model_names]

    bars = ax.barh(model_names, values, xerr=std_vals,
                   color=colors, edgecolor='white', capsize=5)
    ax.set_title(label)
    ax.set_xlim(min(values) - 0.05, 1.0)

    for bar, val in zip(bars, values):
        ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

best_model_name = max(results, key=lambda k: results[k]['auc_mean'])
print(f'\n🏆 Best model by AUC: {best_model_name} '
      f'(AUC = {results[best_model_name]["auc_mean"]:.4f})')

In [ ]:
# ════════════════════════════════════════════════════════════════
# 6.6  FINAL MODEL — TRAIN ON FULL TRAIN, EVALUATE ON TEST
# ════════════════════════════════════════════════════════════════

# Train best model on full training set
final_model = models[best_model_name]
final_model.fit(X_train, y_train)

# Predictions
y_pred      = final_model.predict(X_test)
y_pred_prob = final_model.predict_proba(X_test)[:, 1]

# Metrics
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_prob)

print(f'\n🏆 FINAL MODEL: {best_model_name}')
print('=' * 45)
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1 Score  : {f1:.4f}')
print(f'  ROC-AUC   : {auc:.4f}')
print('=' * 45)
print()
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

In [ ]:
# ── Final Evaluation Plots ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Final Model Evaluation — {best_model_name}', fontsize=14, fontweight='bold')

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# 2. ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.02])

# 3. Churn Probability Distribution
axes[2].hist(y_pred_prob[y_test == 0], bins=40, alpha=0.6,
             color='steelblue', label='No Churn', edgecolor='white')
axes[2].hist(y_pred_prob[y_test == 1], bins=40, alpha=0.6,
             color='tomato', label='Churn', edgecolor='white')
axes[2].axvline(0.5, color='black', linestyle='--', label='Threshold=0.5')
axes[2].set_xlabel('Predicted Churn Probability')
axes[2].set_ylabel('Count')
axes[2].set_title('Predicted Probability Distribution')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════
# 6.7  FEATURE IMPORTANCE — WHAT DRIVES CHURN?
# ════════════════════════════════════════════════════════════════

# Get feature importance from model
if hasattr(final_model[-1], 'feature_importances_'):
    importances = final_model[-1].feature_importances_
elif hasattr(final_model[-1], 'coef_'):
    importances = np.abs(final_model[-1].coef_[0])
else:
    importances = np.zeros(len(feature_cols))

imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=False)

# Plot top 25
top25 = imp_df.head(25)
plt.figure(figsize=(12, 9))
plt.barh(top25['Feature'][::-1], top25['Importance'][::-1],
         color='steelblue', edgecolor='white')
plt.title(f'Top 25 Feature Importances — {best_model_name}',
          fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('\n🔍 Top 25 Features Driving Churn Prediction:')
print('-' * 55)
for i, (_, row) in enumerate(imp_df.head(25).iterrows(), 1):
    bar = '█' * int(row['Importance'] * 200)
    print(f'  {i:2d}. {row["Feature"]:35s} {row["Importance"]:.4f} {bar}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# 6.8  BUSINESS ANALYSIS — OPTIMAL THRESHOLD & SAVINGS
# ════════════════════════════════════════════════════════════════

# Telecom business context:
#   Cost of false negative (missed churn): lose a customer worth $500/year
#   Cost of false positive (wrong alert): waste $50 discount campaign

FN_COST = 500   # lost customer lifetime value
FP_COST = 50    # wasted intervention
TP_SAVE = 400   # successful retention value (partial)

thresholds = np.arange(0.1, 0.9, 0.02)
business_values = []
precision_vals, recall_vals, f1_vals = [], [], []

for thresh in thresholds:
    preds = (y_pred_prob >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()

    net_value = tp * TP_SAVE - fp * FP_COST - fn * FN_COST
    business_values.append(net_value)

    precision_vals.append(precision_score(y_test, preds, zero_division=0))
    recall_vals.append(recall_score(y_test, preds, zero_division=0))
    f1_vals.append(f1_score(y_test, preds, zero_division=0))

optimal_idx = np.argmax(business_values)
optimal_threshold = thresholds[optimal_idx]
max_value = business_values[optimal_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Optimal Threshold — Business Value Analysis', fontsize=13, fontweight='bold')

# Business value curve
axes[0].plot(thresholds, business_values, color='steelblue', lw=2)
axes[0].axvline(optimal_threshold, color='red', linestyle='--',
                label=f'Optimal = {optimal_threshold:.2f}')
axes[0].axhline(0, color='black', linestyle='-', lw=0.5)
axes[0].fill_between(thresholds, business_values,
                      where=np.array(business_values) > 0,
                      alpha=0.2, color='green', label='Positive net value')
axes[0].set_xlabel('Decision Threshold')
axes[0].set_ylabel('Net Business Value ($)')
axes[0].set_title('Business Value vs Threshold')
axes[0].legend()

# Precision-Recall-F1 vs threshold
axes[1].plot(thresholds, precision_vals, label='Precision', color='blue', lw=2)
axes[1].plot(thresholds, recall_vals,    label='Recall',    color='green', lw=2)
axes[1].plot(thresholds, f1_vals,        label='F1 Score',  color='red', lw=2)
axes[1].axvline(optimal_threshold, color='black', linestyle='--',
                label=f'Optimal = {optimal_threshold:.2f}')
axes[1].set_xlabel('Decision Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Precision / Recall / F1 vs Threshold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\n💰 BUSINESS ANALYSIS RESULTS:')
print(f'   Optimal threshold : {optimal_threshold:.2f}')
print(f'   Max net value     : ${max_value:,.0f} (on {len(y_test):,} test customers)')
print(f'   Annualised saving  : ${max_value * 12:,.0f} (extrapolated)')

# Final predictions at optimal threshold
final_preds = (y_pred_prob >= optimal_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, final_preds).ravel()
print(f'\n   At threshold {optimal_threshold:.2f}:')
print(f'   True Positives  (correctly flagged churners)  : {tp:4d}')
print(f'   False Positives (unnecessary interventions)   : {fp:4d}')
print(f'   True Negatives  (correctly kept)              : {tn:4d}')
print(f'   False Negatives (missed churners)             : {fn:4d}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# 6.9  COMPLETE PROJECT SUMMARY
# ════════════════════════════════════════════════════════════════

print('=' * 65)
print('   CUSTOMER CHURN PREDICTION — PROJECT COMPLETE')
print('=' * 65)
print()
print('📦 Dataset')
print(f'   Rows: {raw_df.shape[0]:,}   Original features: {raw_df.shape[1]}')
print(f'   After feature engineering: {X.shape[1]} features')
print()
print('🔧 Feature Engineering Applied')
steps = [
    'Missing value indicators + median imputation',
    'Log transforms (monthly charges, total charges, tenure)',
    'Charge-based ratios (avg per service, total vs expected)',
    'Tenure bands + is_new / is_loyal flags',
    'Service count + protection bundle features',
    'Age binning + senior / young flags',
    'Support call risk + satisfaction flags',
    'Cross-feature interactions (fiber×month-to-month, etc.)',
    'State-level aggregation features',
    'Ordinal encoding (contract), OHE (internet, payment),',
    'Frequency encoding (state)',
]
for step in steps:
    print(f'   ✅ {step}')
print()
print('🤖 Models Compared')
for name, res in sorted(results.items(), key=lambda x: -x[1]['auc_mean']):
    badge = ' 🏆 BEST' if name == best_model_name else ''
    print(f'   {name:<25} AUC={res["auc_mean"]:.4f}{badge}')
print()
print('📊 Final Test Set Performance')
print(f'   ROC-AUC   : {auc:.4f}')
print(f'   F1 Score  : {f1:.4f}')
print(f'   Precision : {prec:.4f}')
print(f'   Recall    : {rec:.4f}')
print(f'   Accuracy  : {acc:.4f}')
print()
print('💰 Business Impact')
print(f'   Optimal threshold : {optimal_threshold:.2f}')
print(f'   Estimated savings : ${max_value:,.0f} per {len(y_test):,} customers')
print()
print('🔑 Top 5 Churn Drivers')
for i, (_, row) in enumerate(imp_df.head(5).iterrows(), 1):
    print(f'   {i}. {row["Feature"]} (importance={row["Importance"]:.4f})')
print()
print('=' * 65)
print('  🎓 Feature Engineering Masterclass — COMPLETE!')
print('=' * 65)

---
## 📋 The Feature Engineering 80/20 Cheat Sheet

| Priority | Technique | When | Impact |
|----------|-----------|------|--------|
| ⭐⭐⭐⭐⭐ | Fix data types | Always | Critical |
| ⭐⭐⭐⭐⭐ | Handle missing values + indicator | Always | Critical |
| ⭐⭐⭐⭐⭐ | Encode categoricals correctly | Always | Critical |
| ⭐⭐⭐⭐ | Log-transform skewed features | Right-skewed data | High |
| ⭐⭐⭐⭐ | Fit scaler on train only (use Pipeline) | Linear/NN models | High |
| ⭐⭐⭐⭐ | Domain-based ratio features | Tabular data | High |
| ⭐⭐⭐ | DateTime decomposition | Time data | Medium |
| ⭐⭐⭐ | Cyclical encoding (month, hour) | Periodic features | Medium |
| ⭐⭐⭐ | Group aggregation features | Entity tables | Medium |
| ⭐⭐ | Remove high-correlation features | > 0.9 corr | Medium |
| ⭐⭐ | Tree importance / RFE selection | Post-training | Medium |
| ⭐ | Interaction features | Domain knowledge | Situational |

---

## 📚 Next Steps

1. **Practice:** [Kaggle — Titanic](https://kaggle.com/c/titanic), [House Prices](https://kaggle.com/c/house-prices-advanced-regression-techniques), [Give Me Some Credit](https://kaggle.com/c/GiveMeSomeCredit)
2. **Libraries:** `feature-engine`, `featuretools` (automated FE), `category_encoders`
3. **Advanced:** PCA/t-SNE, embeddings for high-cardinality, AutoML FE
4. **Book:** *Feature Engineering for Machine Learning* — Alice Zheng & Amanda Casari (O'Reilly)

---
*Feature Engineering Masterclass | Complete Notebook | GitHub-Ready*